# EXP-008 - EgyBERT Joint L1+L2+L3 Classification (Kaggle T4)

**Purpose:** Fine-tune `faisalq/EgyBERT` on the ArabicITSM-9K dataset  
using the identical three-task (L1+L2+L3) configuration as EXP-006a (MARBERTv2).  
The only variable that changes is the pretrained encoder; all hyperparameters,  
data splits, preprocessing, and architecture are identical to EXP-006a.

**Comparison target:** `kaggle_train_l1l2l3_arabic_itsm_classification.ipynb` (EXP-006a)  
**Model:** `faisalq/EgyBERT` (Egyptian-dialect-pretrained encoder)  
**Tasks:** L1 (6 classes) + L2 (16 classes) + L3 (48 classes)  
**Hardware:** Kaggle Tesla T4 (16 GB VRAM)  
**Output:** `models/egybert_l1_l2_l3_best/`

In [ ]:
# Clone the repository and install dependencies
# Mirrors the setup cell in kaggle_train_l1l2l3_arabic_itsm_classification.ipynb exactly
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding Kaggle's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml

In [ ]:
# Copy processed data from Kaggle dataset input
# Mirrors kaggle_train_l1l2l3_arabic_itsm_classification.ipynb exactly
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/

!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

In [ ]:
!nvidia-smi

In [ ]:
# EXP-008: EgyBERT joint L1+L2+L3 training
#
# Identical to EXP-006a (MARBERTv2) except for --model.
# All other arguments are unchanged:
#   --tasks l1 l2 l3  : three-task joint training
#   --epochs 10       : same budget as EXP-006a
#   --lr 1e-5         : same learning rate as EXP-006a
#   --batch-size 16   : same batch size as EXP-006a
#
# Note: train.py may save either marbert_{task_label}_best or egybert_{task_label}_best,
# depending on the repo version used in Kaggle. The next cell normalizes the folder name.

!python scripts/train.py \\
    --model faisalq/EgyBERT \\
    --tasks l1 l2 l3 \\
    --data-dir data/processed \\
    --epochs 10 \\
    --lr 1e-5 \\
    --batch-size 16

In [ ]:
import os

# Normalize the checkpoint name to egybert_ for clarity.
src = 'models/marbert_l1_l2_l3_best'
dst = 'models/egybert_l1_l2_l3_best'

if os.path.exists(src) and not os.path.exists(dst):
    os.rename(src, dst)
    print(f'Renamed: {src} -> {dst}')
elif os.path.exists(dst):
    print(f'Already exists: {dst}')
else:
    raise FileNotFoundError(f'Expected checkpoint not found: {src}')

In [ ]:
import os, shutil, torch

checkpoint_dir = 'models/egybert_l1_l2_l3_best'
print('Checkpoint files:', os.listdir(checkpoint_dir))

# Verify heads.pt contains exactly the 3 expected task heads
heads = torch.load(f'{checkpoint_dir}/heads.pt', map_location='cpu', weights_only=False)
print('Head keys:', sorted(heads.keys()))

expected_keys = {'l1.1.weight', 'l1.1.bias', 'l2.1.weight', 'l2.1.bias', 'l3.1.weight', 'l3.1.bias'}
assert expected_keys == set(heads.keys()), f'Unexpected keys: {set(heads.keys())}'
print('\\nVerification passed: 3 joint heads found (l1, l2, l3)')

# Create download archive
shutil.make_archive('/kaggle/working/egybert_l1_l2_l3_best', 'zip', 'models', 'egybert_l1_l2_l3_best')
print('\\nDownload archive: /kaggle/working/egybert_l1_l2_l3_best.zip')
print('Place in: arabic-itsm-classification/models/egybert_l1_l2_l3_best/')

## Next Steps

1. Download `egybert_l1_l2_l3_best.zip` from Kaggle output
2. Extract to `arabic-itsm-classification/models/egybert_l1_l2_l3_best/`
3. Evaluate the checkpoint on the fixed 1,433-ticket test split and record L1/L2/L3 macro-F1
4. Add the resulting scores to the paper's encoder-comparison table and discussion